# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [3]:
# Exploración básica del dataset
print(f"Dimensiones: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nEstadísticas descriptivas:")
from IPython.display import display
display(df.describe())
df.head()

Dimensiones: (912, 13)

Tipos de datos:
laptop_ID             int64
Company                 str
Product                 str
TypeName                str
Inches              float64
ScreenResolution        str
Cpu                     str
Ram                     str
Memory                  str
Gpu                     str
OpSys                   str
Weight                  str
Price_in_euros      float64
dtype: object

Valores nulos por columna:
laptop_ID           0
Company             0
Product             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price_in_euros      0
dtype: int64

Estadísticas descriptivas:


,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


### 2.2 Definir X e y


In [4]:
import re
from IPython.display import display

def extract_features(df_raw):
    df = df_raw.copy()

    # RAM: '8GB' -> 8
    df['ram_gb'] = df['Ram'].str.replace('GB', '').astype(int)

    # Peso: '1.86kg' -> 1.86
    df['weight_kg'] = df['Weight'].str.replace('kg', '').astype(float)

    # Resolución de pantalla
    res = df['ScreenResolution'].str.extract(r'(\d{3,4})x(\d{3,4})')
    df['res_width'] = res[0].astype(int)
    df['res_height'] = res[1].astype(int)
    df['is_touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
    df['is_ips'] = df['ScreenResolution'].str.contains('IPS').astype(int)
    df['ppi'] = np.sqrt(df['res_width']**2 + df['res_height']**2) / df['Inches']

    # CPU: marca y velocidad
    df['cpu_brand'] = df['Cpu'].apply(
        lambda x: 'Intel' if 'Intel' in x else ('AMD' if 'AMD' in x else 'Other')
    )
    df['cpu_ghz'] = df['Cpu'].str.extract(r'(\d+\.?\d*)GHz')[0].astype(float)
    df['cpu_ghz'] = df['cpu_ghz'].fillna(df['cpu_ghz'].median())

    # GPU: marca
    df['gpu_brand'] = df['Gpu'].apply(
        lambda x: 'Nvidia' if 'Nvidia' in x
        else ('AMD' if 'AMD' in x else ('Intel' if 'Intel' in x else 'Other'))
    )

    # Memoria: tipo y capacidad total en GB
    def parse_memory(mem):
        total_gb = 0
        for part in mem.split('+'):
            for val, unit in re.findall(r'(\d+\.?\d*)(TB|GB)', part):
                total_gb += float(val) * 1000 if unit == 'TB' else float(val)
        if 'SSD' in mem:
            mem_type = 'SSD'
        elif 'Flash' in mem:
            mem_type = 'Flash'
        elif 'Hybrid' in mem:
            mem_type = 'Hybrid'
        else:
            mem_type = 'HDD'
        return total_gb, mem_type

    mem_parsed = df['Memory'].apply(parse_memory)
    df['memory_gb'] = mem_parsed.apply(lambda x: x[0])
    df['memory_type'] = mem_parsed.apply(lambda x: x[1])

    return df


# Aplicar feature engineering
df_fe = extract_features(df)

# Columnas que usará el modelo
feature_cols = ['Company', 'TypeName', 'Inches', 'ram_gb', 'weight_kg',
                'res_width', 'res_height', 'is_touchscreen', 'is_ips', 'ppi',
                'cpu_brand', 'cpu_ghz', 'gpu_brand', 'memory_gb', 'memory_type',
                'OpSys']

X = df_fe[feature_cols].copy()
y = df_fe['Price_in_euros'].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

X shape: (912, 16)
y shape: (912,)


,Company,TypeName,Inches,ram_gb,weight_kg,res_width,res_height,is_touchscreen,is_ips,ppi,cpu_brand,cpu_ghz,gpu_brand,memory_gb,memory_type,OpSys
0,HP,Notebook,15.6,8,1.86,1920,1080,0,0,141.211998,Intel,2.0,Intel,256.0,SSD,Windows 10
1,Dell,Gaming,15.6,16,2.59,1920,1080,0,0,141.211998,Intel,2.6,Nvidia,1000.0,HDD,Windows 10
2,HP,Notebook,15.6,8,2.04,1920,1080,0,0,141.211998,Intel,2.7,Nvidia,1000.0,HDD,Windows 10
3,Apple,Ultrabook,13.3,8,1.34,1440,900,0,0,127.677940,Intel,1.8,Intel,128.0,Flash,macOS
4,Dell,Notebook,15.6,4,2.25,1920,1080,0,0,141.211998,Intel,2.0,AMD,1000.0,HDD,Linux


### 2.3 Dividir en train y test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (729, 16)
X_test:  (183, 16)


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [6]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['Company', 'TypeName', 'OpSys', 'cpu_brand', 'gpu_brand', 'memory_type']

# fit SOLO sobre X_train (evitar data leakage); unknown_value=-1 gestiona categorias nuevas en test
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train = X_train.copy()
X_test = X_test.copy()

X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

print("Encoding aplicado correctamente.")
X_train.head()

Encoding aplicado correctamente.


,Company,TypeName,Inches,ram_gb,weight_kg,res_width,res_height,is_touchscreen,is_ips,ppi,cpu_brand,cpu_ghz,gpu_brand,memory_gb,memory_type,OpSys
25,7.0,5.0,17.3,8,3.00,1920,1080,0,1,127.335675,1.0,2.6,0.0,1000.0,1.0,7.0
84,4.0,1.0,15.6,16,2.56,1920,1080,0,0,141.211998,1.0,2.8,2.0,512.0,3.0,5.0
10,1.0,4.0,13.3,8,1.37,2560,1600,0,1,226.983005,1.0,2.9,1.0,512.0,3.0,8.0
342,7.0,3.0,14.0,4,1.54,1920,1080,0,0,157.350512,1.0,2.3,1.0,500.0,1.0,7.0
890,4.0,3.0,17.3,16,2.80,1920,1080,0,0,127.335675,1.0,1.8,0.0,2256.0,3.0,5.0


## 4. Modelado

### 4.1 Entrenamiento

In [7]:
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Modelo RandomForestRegressor entrenado.")

Modelo RandomForestRegressor entrenado.


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [8]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
rmse_test  = root_mean_squared_error(y_test, y_pred_test)

print(f"RMSE Train: {rmse_train:.2f} euros")
print(f"RMSE Test:  {rmse_test:.2f} euros")

RMSE Train: 111.02 euros
RMSE Test:  360.33 euros


### 4.3 Optimización (up to you 🫰🏻)

In [9]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

best_params = search.best_params_
print(f"Mejores parametros: {best_params}")
print(f"Mejor RMSE en CV:   {-search.best_score_:.2f} euros")

y_pred_best = search.best_estimator_.predict(X_test)
rmse_best = root_mean_squared_error(y_test, y_pred_best)
print(f"RMSE Test (modelo optimizado): {rmse_best:.2f} euros")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Mejores parametros: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}
Mejor RMSE en CV:   289.13 euros
RMSE Test (modelo optimizado): 318.29 euros


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [10]:
# Feature engineering sobre el dataset completo de train
df_fe_full = extract_features(df)
X_full = df_fe_full[feature_cols].copy()
y_full = df_fe_full['Price_in_euros'].copy()

# OrdinalEncoder entrenado con el 100% de los datos de train
oe_full = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_full[cat_cols] = oe_full.fit_transform(X_full[cat_cols])

# Entrenar modelo final con los mejores hiperparametros encontrados
model_final = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
model_final.fit(X_full, y_full)

print(f"Modelo final entrenado con {X_full.shape[0]} muestras.")

Modelo final entrenado con 912 muestras.


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [11]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [12]:
# Feature engineering sobre test (mismos pasos que en train)
X_pred_fe = extract_features(X_pred)
X_pred_proc = X_pred_fe[feature_cols].copy()

# Aplicar .transform() con el encoder ajustado sobre el train completo
X_pred_proc[cat_cols] = oe_full.transform(X_pred_proc[cat_cols])

print(f"Test procesado: {X_pred_proc.shape}")
X_pred_proc.head()

Test procesado: (391, 16)


,Company,TypeName,Inches,ram_gb,weight_kg,res_width,res_height,is_touchscreen,is_ips,ppi,cpu_brand,cpu_ghz,gpu_brand,memory_gb,memory_type,OpSys
0,10.0,1.0,15.6,16,2.400,1920,1080,0,0,141.211998,1.0,2.8,2.0,512.0,3.0,4.0
1,0.0,3.0,15.6,4,2.400,1366,768,0,0,100.454670,1.0,1.6,1.0,500.0,1.0,2.0
2,10.0,3.0,15.6,4,1.900,1366,768,0,0,100.454670,1.0,2.0,1.0,1000.0,1.0,4.0
3,4.0,0.0,15.6,8,2.191,1920,1080,1,1,141.211998,1.0,2.5,1.0,256.0,3.0,5.0
4,7.0,3.0,14.0,4,1.950,1920,1080,0,0,157.350512,1.0,2.5,1.0,256.0,3.0,5.0


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [13]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [14]:
y_pred_final = model_final.predict(X_pred_proc)

submission = pd.DataFrame({
    'laptop_ID': X_pred['laptop_ID'],
    'Price_in_euros': y_pred_final
})

print(f"Submission shape: {submission.shape}")
submission.head()

Submission shape: (391, 2)


,laptop_ID,Price_in_euros
0,209,1238.152500
1,1281,297.123842
2,1168,379.928500
3,1231,1037.619540
4,1020,1024.710500


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [15]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [16]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260629_092008.csv'. ¡A Kaggle!
